# 03. Factor selection

This notebook compares common-eps fixed-eigenmode AR(1)--Binomial models with `R = 1, 2, 3, 4` and selects `R = 2` as the parsimonious main specification.

In [ ]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
from scipy.special import expit, logit
from scipy.stats import binom, betabinom, norm, rankdata, spearmanr, kendalltau
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT = Path(".")
DATA = ROOT / "data"
PDATA = ROOT / "pdata"
FIGURES = ROOT / "figures"
TABLES = ROOT / "tables"
for directory in [DATA, PDATA, FIGURES, TABLES]:
    directory.mkdir(exist_ok=True)

RNG = np.random.default_rng(20260618)
plt.style.use("default")


SECTOR_COLORS = {
    "Basic Materials": "#4C78A8",
    "Capital Goods": "#F58518",
    "Consumer": "#54A24B",
    "Energy": "#E45756",
    "Finance": "#72B7B2",
    "Healthcare": "#B279A2",
    "Technology": "#FF9DA6",
    "Utilities": "#9D755D",
}


def read_sample():
    df = pd.read_csv(DATA / "SAMPLE.csv", parse_dates=["date"])
    expected = ["date", "sector", "obligors", "defaulted"]
    if list(df.columns) != expected:
        raise ValueError(f"Expected columns {expected}, got {list(df.columns)}")
    df = df.sort_values(["date", "sector"]).reset_index(drop=True)
    if (df["obligors"] <= 0).any():
        raise ValueError("obligors must be positive")
    if ((df["defaulted"] < 0) | (df["defaulted"] > df["obligors"])).any():
        raise ValueError("defaulted must lie between zero and obligors")
    return df


def empirical_logit(defaulted, obligors, smooth=0.5):
    defaulted = np.asarray(defaulted, dtype=float)
    obligors = np.asarray(obligors, dtype=float)
    return np.log((defaulted + smooth) / (obligors - defaulted + smooth))


def panel_matrices(df):
    dates = pd.Index(sorted(df["date"].unique()), name="date")
    sectors = pd.Index(sorted(df["sector"].unique()), name="sector")
    dmat = (
        df.pivot(index="date", columns="sector", values="defaulted")
        .reindex(index=dates, columns=sectors)
        .astype(float)
    )
    nmat = (
        df.pivot(index="date", columns="sector", values="obligors")
        .reindex(index=dates, columns=sectors)
        .astype(float)
    )
    x = pd.DataFrame(
        empirical_logit(dmat.to_numpy(), nmat.to_numpy()),
        index=dates,
        columns=sectors,
    )
    rates = dmat / nmat
    return dates, sectors, dmat, nmat, x, rates


def aggregate_k_month(df, k):
    wide_dates = pd.Index(sorted(df["date"].unique()))
    block_lookup = {date: i // k for i, date in enumerate(wide_dates)}
    tmp = df.copy()
    tmp["block"] = tmp["date"].map(block_lookup)
    tmp["block_start"] = tmp["block"].map(lambda b: wide_dates[b * k])
    out = (
        tmp.groupby(["block", "block_start", "sector"], as_index=False)
        .agg(obligors=("obligors", "sum"), defaulted=("defaulted", "sum"))
    )
    out = out.rename(columns={"block_start": "date"})
    out["k_month"] = k
    return out[["date", "sector", "k_month", "obligors", "defaulted"]]


def fit_common_eps_model(df, R=2):
    dates, sectors, dmat, nmat, x, rates = panel_matrices(df)
    X = x.to_numpy()
    intercept = X.mean(axis=0)
    Xc = X - intercept
    cov = np.cov(Xc, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    loadings = eigvecs[:, :R]
    scores = Xc @ loadings
    phis = []
    innov_sd = []
    for r in range(R):
        y = scores[1:, r]
        z = scores[:-1, r]
        denom = float(np.dot(z, z))
        phi = float(np.dot(z, y) / denom) if denom > 1e-12 else 0.0
        phi = float(np.clip(phi, -0.98, 0.98))
        resid = y - phi * z
        phis.append(phi)
        innov_sd.append(float(np.sqrt(np.mean(resid**2) + 1e-8)))
    fitted_centered = scores @ loadings.T
    residual = Xc - fitted_centered
    sigma_eps_common = float(np.sqrt(np.mean(residual**2) + 1e-8))
    eta_hat = intercept + fitted_centered
    p_hat = expit(eta_hat)
    ll = float(binom.logpmf(dmat.to_numpy(), nmat.to_numpy(), p_hat).sum())
    n_params = len(sectors) + R * (len(sectors) + 2) + 1
    aic = float(2 * n_params - 2 * ll)
    return {
        "dates": list(dates),
        "sectors": list(sectors),
        "defaulted": dmat,
        "obligors": nmat,
        "logit": x,
        "rates": rates,
        "intercept": intercept,
        "eigvals": eigvals,
        "loadings": loadings,
        "scores": scores,
        "phis": np.array(phis),
        "innov_sd": np.array(innov_sd),
        "sigma_eps_common": sigma_eps_common,
        "eta_hat": eta_hat,
        "p_hat": p_hat,
        "log_likelihood": ll,
        "aic": aic,
    }


def acf_values(x, max_lag=24):
    x = np.asarray(x, dtype=float)
    x = x - np.nanmean(x)
    denom = np.dot(x, x)
    if denom <= 1e-12:
        return np.zeros(max_lag + 1)
    vals = [1.0]
    for lag in range(1, max_lag + 1):
        vals.append(float(np.dot(x[:-lag], x[lag:]) / denom))
    return np.array(vals)


def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

In [ ]:
df = read_sample()
all_dates = sorted(df["date"].unique())
holdout_months = 36
cutoff = all_dates[-holdout_months]
train = df[df["date"] < cutoff].copy()
test = df[df["date"] >= cutoff].copy()
_, sectors, dtest, ntest, xtest, _ = panel_matrices(test)

rows = []
for R in [1, 2, 3, 4]:
    model = fit_common_eps_model(train, R=R)
    centered_test = xtest.to_numpy() - model["intercept"]
    projected_scores = centered_test @ model["loadings"][:, :R]
    eta_test = model["intercept"] + projected_scores @ model["loadings"][:, :R].T
    p_test = np.clip(expit(eta_test), 1e-8, 1 - 1e-8)
    holdout_ll = float(binom.logpmf(dtest.to_numpy(), ntest.to_numpy(), p_test).sum())
    rows.append({
        "R": R,
        "train_log_likelihood": model["log_likelihood"],
        "train_aic": model["aic"],
        "holdout_log_score": holdout_ll,
        "sigma_eps_common": model["sigma_eps_common"],
        "pc_variance_share": float(model["eigvals"][:R].sum() / model["eigvals"].sum()),
    })

selection = pd.DataFrame(rows)
selection["delta_holdout_log_score"] = selection["holdout_log_score"] - selection["holdout_log_score"].max()
selection["selected_main_specification"] = selection["R"].eq(2)
selection.to_csv(TABLES / "table2_common_eps_factor_selection.csv", index=False)
selection.to_csv(PDATA / "factor_selection.csv", index=False)
(PDATA / "selected_factor_dimension.json").write_text(json.dumps({
    "selected_R": 2,
    "reason": "R=2 retains the market-wide and sector-rotation factors while remaining parsimonious and stable."
}, indent=2), encoding="utf-8")
selection

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.5))
axes[0].plot(selection["R"], selection["holdout_log_score"], marker="o", color="#4C78A8", label="Holdout log score")
ax2 = axes[0].twinx()
ax2.plot(selection["R"], selection["train_aic"], marker="s", color="#E45756", label="Train AIC")
axes[0].axvline(2, color="black", ls="--", lw=1)
axes[0].set_xlabel("Number of dynamic factors R")
axes[0].set_ylabel("Holdout log score")
ax2.set_ylabel("Train AIC")
axes[0].set_title("Fit and holdout diagnostics")
lines = [line for line in axes[0].get_lines() + ax2.get_lines() if not line.get_label().startswith("_")]
axes[0].legend(lines, [line.get_label() for line in lines], fontsize=8)

axes[1].plot(selection["R"], selection["sigma_eps_common"], marker="o", label="Common residual scale", color="#72B7B2")
axes[1].plot(selection["R"], selection["pc_variance_share"], marker="s", label="PC variance share", color="#F58518")
axes[1].axvline(2, color="black", ls="--", lw=1)
axes[1].set_xlabel("Number of dynamic factors R")
axes[1].set_title("Parsimony diagnostics")
axes[1].legend(fontsize=8)
for ax in axes:
    ax.set_xticks([1, 2, 3, 4])
fig.suptitle("Figure 4. Common-eps factor selection")
savefig(FIGURES / "figure4_factor_selection.png")